# Mango Demand Forecasting - Data Preprocessing

This notebook processes the raw train and test data to create `train_processed.csv` and `test_processed.csv` with engineered features.

## 1. Load Data and Setup

In [1]:
# Load train data
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import os

# Fix for Windows threadpoolctl/OpenBLAS issue (if needed)
os.environ['OMP_NUM_THREADS'] = '1'

# Load data
df = pd.read_csv('data/train.csv', sep=";", header=0)
df_test = pd.read_csv('data/test.csv', sep=";", header=0)

print(f"Train data shape: {df.shape}")
print(f"Test data shape: {df_test.shape}")

Train data shape: (95339, 33)
Test data shape: (2250, 33)


## 2. Basic Data Cleaning

In [2]:
def clean_categorical_feature(df_train, df_test, feature_name):
    """
    Clean a categorical feature by determining if it's applicable to each family.
    - If value exists: keep it
    - If missing but family has this feature: mark as "MISSING_VALUE"
    - If missing and family doesn't have this feature: mark as "NOT_APPLICABLE"
    """
    # Determine applicability from train data
    family_applicable = df_train.groupby("family")[feature_name].apply(
        lambda x: x.notna().any()
    ).to_dict()
    
    def clean_value(row):
        value = row[feature_name]
        applicable = row[f"{feature_name}_applicable"]
        
        if pd.notnull(value):
            return value
        else:
            return "MISSING_VALUE" if applicable == 1 else "NOT_APPLICABLE"
    
    # Apply to train
    df_train[f"{feature_name}_applicable"] = df_train["family"].map(family_applicable).astype(int)
    df_train[feature_name] = df_train.apply(clean_value, axis=1)
    df_train.drop(columns=[f"{feature_name}_applicable"], inplace=True)
    
    # Apply to test using train's applicability mapping
    df_test[f"{feature_name}_applicable"] = df_test["family"].map(family_applicable).fillna(0).astype(int)
    df_test[feature_name] = df_test.apply(clean_value, axis=1)
    df_test.drop(columns=[f"{feature_name}_applicable"], inplace=True)
    
    return df_train, df_test

# Clean categorical features
categorical_features = [
    "waist_type", "length_type", "silhouette_type", 
    "neck_lapel_type", "sleeve_length_type", 
    "woven_structure", "knit_structure"
]

for feature in categorical_features:
    df, df_test = clean_categorical_feature(df, df_test, feature)
    print(f"Cleaned {feature}")

print("\nBasic cleaning complete!")

Cleaned waist_type
Cleaned length_type
Cleaned silhouette_type
Cleaned neck_lapel_type
Cleaned sleeve_length_type
Cleaned woven_structure
Cleaned knit_structure

Basic cleaning complete!


## 3. Create Basic Features


In [3]:
# Create is_fall boolean (even id_season = fall, odd = not fall)
df['is_fall'] = (df['id_season'] % 2 == 0).astype(int)
df_test['is_fall'] = (df_test['id_season'] % 2 == 0).astype(int)

# Fill missing print_type
df["print_type"] = df["print_type"].fillna("Sin Estampado")
df_test["print_type"] = df_test["print_type"].fillna("Sin Estampado")

# Create weeks_since_launch (rank by num_week_iso within each ID and season)
df = df.sort_values(['ID', 'id_season', 'year', 'num_week_iso'])
df['weeks_since_launch'] = (df.groupby(['ID', 'id_season'])['num_week_iso'].rank(method='dense', ascending=True) - 1).astype(int)

# Handle negative sales
df.loc[df["weekly_sales"] < 0, "weekly_sales"] = 0
df.loc[df["weekly_demand"] < 0, "weekly_demand"] = 0

# Add seasonality features
df["is_week_23"] = (df["num_week_iso"] == 23).astype(int)
df["is_black_friday"] = (df["num_week_iso"].isin([47, 48])).astype(int)

if 'num_week_iso' in df_test.columns:
    df_test["is_week_23"] = (df_test["num_week_iso"] == 23).astype(int)
    df_test["is_black_friday"] = (df_test["num_week_iso"].isin([47, 48])).astype(int)
else:
    df_test["is_week_23"] = 0
    df_test["is_black_friday"] = 0

print("Basic features created!")


Basic features created!


## 4. Color Clustering


In [4]:
# Color clustering using color_name with k-means (k=15)
print("Creating color clusters from color_name...")

# Label encode color names
le_color = LabelEncoder()
color_names_train = df["color_name"].fillna("UNKNOWN")
color_encoded = le_color.fit_transform(color_names_train)
color_encoded_2d = color_encoded.reshape(-1, 1)

# Apply k-means clustering
n_color_clusters = 15
kmeans_color = KMeans(n_clusters=n_color_clusters, n_init=10, random_state=42)
color_clusters = kmeans_color.fit_predict(color_encoded_2d)

# Add color cluster to dataframe
df["color_cluster"] = color_clusters.astype(int)
color_dists = kmeans_color.transform(color_encoded_2d)
df["color_cluster_dist"] = color_dists.min(axis=1)

# Apply to test data
test_color_names = df_test["color_name"].fillna("UNKNOWN")
test_color_encoded = []
for color in test_color_names:
    try:
        test_color_encoded.append(le_color.transform([color])[0])
    except ValueError:
        try:
            test_color_encoded.append(le_color.transform(["UNKNOWN"])[0])
        except ValueError:
            test_color_encoded.append(0)

test_color_encoded = np.array(test_color_encoded).reshape(-1, 1)
test_color_clusters = kmeans_color.predict(test_color_encoded)
df_test["color_cluster"] = test_color_clusters.astype(int)
test_color_dists = kmeans_color.transform(test_color_encoded)
df_test["color_cluster_dist"] = test_color_dists.min(axis=1)

# Drop RGB columns
cols_to_drop = ["color_rgb"]
for col in ["R", "G", "B"]:
    if col in df.columns:
        cols_to_drop.append(col)
df.drop(columns=cols_to_drop, inplace=True)
df_test.drop(columns=[col for col in cols_to_drop if col in df_test.columns], inplace=True)

print(f"Created {n_color_clusters} color clusters")


Creating color clusters from color_name...
Created 15 color clusters


## 5. Image Embedding Processing

In [5]:
# Parse embeddings
import ast

def parse_embedding(x):
    """Robustly convert string/list/array embedding into np.ndarray."""
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x)
    if x is None:
        return None
    try:
        parsed = ast.literal_eval(str(x))
        return np.array(parsed)
    except Exception:
        return None

df["embedding_array"] = df["image_embedding"].apply(parse_embedding)
valid = df["embedding_array"].notna()
emb_matrix = np.vstack(df.loc[valid, "embedding_array"].values)
print(f"Embeddings shape: {emb_matrix.shape}")

# Apply PCA (83 components)
scaler_emb = StandardScaler()
emb_scaled = scaler_emb.fit_transform(emb_matrix)
n_components = 83
pca = PCA(n_components=n_components)
emb_pca = pca.fit_transform(emb_scaled)

# Add PCA features to dataframe
for i in range(n_components):
    df.loc[valid, f"emb_pca_{i+1}"] = emb_pca[:, i]
    df.loc[~valid, f"emb_pca_{i+1}"] = 0

print(f"Created {n_components} PCA features")

# Clustering with PCA (22 clusters)
n_clusters = 22
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
clusters = kmeans.fit_predict(emb_pca)
df.loc[valid, "emb_cluster"] = clusters
df.loc[~valid, "emb_cluster"] = -1
df["emb_cluster"] = df["emb_cluster"].astype(int)
dists = kmeans.transform(emb_pca)
df.loc[valid, "emb_dist"] = dists.min(axis=1)
df.loc[~valid, "emb_dist"] = -1

# Apply to test data
df_test["embedding_array"] = df_test["image_embedding"].apply(parse_embedding)
test_valid = df_test["embedding_array"].notna()

if test_valid.sum() > 0:
    test_emb_matrix = np.vstack(df_test.loc[test_valid, "embedding_array"].values)
    test_emb_scaled = scaler_emb.transform(test_emb_matrix)
    test_emb_pca = pca.transform(test_emb_scaled)
    test_clusters = kmeans.predict(test_emb_pca)
    
    for i in range(n_components):
        df_test.loc[test_valid, f"emb_pca_{i+1}"] = test_emb_pca[:, i]
        df_test.loc[~test_valid, f"emb_pca_{i+1}"] = 0
    
    df_test.loc[test_valid, "emb_cluster"] = test_clusters
    df_test.loc[~test_valid, "emb_cluster"] = -1
    df_test["emb_cluster"] = df_test["emb_cluster"].astype(int)
    test_dists = kmeans.transform(test_emb_pca)
    df_test.loc[test_valid, "emb_dist"] = test_dists.min(axis=1)
    df_test.loc[~test_valid, "emb_dist"] = -1
else:
    for i in range(n_components):
        df_test[f"emb_pca_{i+1}"] = 0
    df_test["emb_cluster"] = -1
    df_test["emb_dist"] = -1

print(f"Using {n_clusters} embedding clusters")


Embeddings shape: (95339, 512)
Created 83 PCA features
Using 22 embedding clusters


/var/folders/gy/5nns71_n7nx_v_2mkns5_k340000gn/T/ipykernel_1985/967163749.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test.loc[test_valid, f"emb_pca_{i+1}"] = test_emb_pca[:, i]
/var/folders/gy/5nns71_n7nx_v_2mkns5_k340000gn/T/ipykernel_1985/967163749.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test.loc[test_valid, f"emb_pca_{i+1}"] = test_emb_pca[:, i]
/var/folders/gy/5nns71_n7nx_v_2mkns5_k340000gn/T/ipykernel_1985/967163749.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

## 6. Feature Engineering

### 6.1 Cluster Velocity Feature


In [6]:
# Compute velocity_1_3: average sales per week during first 3 weeks for each cluster
print("Computing cluster-level trend features...")
train_first_3_weeks = df[df['weeks_since_launch'] < 3].copy()
cluster_velocity = train_first_3_weeks.groupby('emb_cluster')['weekly_sales'].mean().reset_index()
cluster_velocity.columns = ['emb_cluster', 'velocity_1_3']

overall_velocity = train_first_3_weeks['weekly_sales'].mean()
if -1 not in cluster_velocity['emb_cluster'].values:
    cluster_velocity = pd.concat([
        cluster_velocity,
        pd.DataFrame([{'emb_cluster': -1, 'velocity_1_3': overall_velocity}])
    ], ignore_index=True)

df = df.merge(cluster_velocity, on='emb_cluster', how='left')
df['velocity_1_3'] = df['velocity_1_3'].fillna(overall_velocity)
df_test = df_test.merge(cluster_velocity, on='emb_cluster', how='left')
df_test['velocity_1_3'] = df_test['velocity_1_3'].fillna(overall_velocity)

print(f"Velocity range: {cluster_velocity['velocity_1_3'].min():.2f} to {cluster_velocity['velocity_1_3'].max():.2f}")

Computing cluster-level trend features...
Velocity range: 459.79 to 1469.42


### 6.2 Trend Score Feature

In [7]:
# Engineer trend score: similarity to top/bottom performers
print("Engineering trend score feature...")

# Identify top 20% and bottom 20% products by total sales
product_sales = df.groupby('ID')['weekly_sales'].sum().reset_index()
product_sales.columns = ['ID', 'total_weekly_sales']
top_threshold = product_sales['total_weekly_sales'].quantile(0.8)
bottom_threshold = product_sales['total_weekly_sales'].quantile(0.2)
top_product_ids = product_sales[product_sales['total_weekly_sales'] >= top_threshold]['ID'].values
bottom_product_ids = product_sales[product_sales['total_weekly_sales'] <= bottom_threshold]['ID'].values

# Get unique products with their embeddings
train_products = df.groupby('ID').first().reset_index()
pca_cols = [f'emb_pca_{i+1}' for i in range(n_components)]
top_train_products = train_products[train_products['ID'].isin(top_product_ids)]
bottom_train_products = train_products[train_products['ID'].isin(bottom_product_ids)]

# Compute centroid embeddings
top_centroid = top_train_products[pca_cols].mean().values
bottom_centroid = bottom_train_products[pca_cols].mean().values

# Compute similarity to centroids
def compute_similarity_to_centroid(embedding_values, centroid):
    if hasattr(embedding_values, 'values'):
        embedding_arr = embedding_values.values
    else:
        embedding_arr = np.array(embedding_values)
    if embedding_arr is None or np.isnan(embedding_arr).any():
        return 0.0
    embedding_2d = embedding_arr.reshape(1, -1)
    centroid_2d = centroid.reshape(1, -1)
    return cosine_similarity(embedding_2d, centroid_2d)[0][0]

train_embeddings = train_products[pca_cols]
train_products['sim_to_top'] = train_embeddings.apply(
    lambda row: compute_similarity_to_centroid(row, top_centroid), axis=1
)
train_products['sim_to_bottom'] = train_embeddings.apply(
    lambda row: compute_similarity_to_centroid(row, bottom_centroid), axis=1
)

# Merge to full dataframes
for col in ['sim_to_top', 'sim_to_bottom', 'trend_score']:
    if col in df.columns:
        df = df.drop(columns=[col])
df = df.merge(train_products[['ID', 'sim_to_top', 'sim_to_bottom']], on='ID', how='left')
df['sim_to_top'] = df['sim_to_top'].fillna(0)
df['sim_to_bottom'] = df['sim_to_bottom'].fillna(0)
df['trend_score'] = df['sim_to_top'] - df['sim_to_bottom']

# Apply to test
test_products = df_test.groupby('ID').first().reset_index()
test_embeddings = test_products[pca_cols]
test_products['sim_to_top'] = test_embeddings.apply(
    lambda row: compute_similarity_to_centroid(row, top_centroid), axis=1
)
test_products['sim_to_bottom'] = test_embeddings.apply(
    lambda row: compute_similarity_to_centroid(row, bottom_centroid), axis=1
)

for col in ['sim_to_top', 'sim_to_bottom', 'trend_score']:
    if col in df_test.columns:
        df_test = df_test.drop(columns=[col])
df_test = df_test.merge(test_products[['ID', 'sim_to_top', 'sim_to_bottom']], on='ID', how='left')
df_test['sim_to_top'] = df_test['sim_to_top'].fillna(0)
df_test['sim_to_bottom'] = df_test['sim_to_bottom'].fillna(0)
df_test['trend_score'] = df_test['sim_to_top'] - df_test['sim_to_bottom']

print("Trend score feature complete!")

Engineering trend score feature...
Trend score feature complete!


### 6.3 Advanced Features (Cluster, Similarity, Family-level) for Train Data

In [8]:
print("=" * 60)
print("Engineering Advanced Features for Train Data")
print("=" * 60)

# ========== CLUSTER-LEVEL FEATURES ==========
print("\n1. Cluster-level features...")

# Cluster velocity 1-6 (mean sales per week during first 6 weeks)
train_first_6_weeks = df[df['weeks_since_launch'] < 6].copy()
cluster_velocity_1_6 = train_first_6_weeks.groupby('emb_cluster')['weekly_sales'].mean().reset_index()
cluster_velocity_1_6.columns = ['emb_cluster', 'cluster_velocity_1_6']

# Cluster demand last season (mean weekly_sales for each cluster in previous season)
print("Computing cluster demand last season...")
cluster_season_means = df.groupby(['emb_cluster', 'id_season'])['weekly_sales'].mean().reset_index()
cluster_season_means.columns = ['emb_cluster', 'id_season', 'cluster_season_mean']

cluster_overall_means = df.groupby('emb_cluster')['weekly_sales'].mean().reset_index()
cluster_overall_means.columns = ['emb_cluster', 'cluster_overall_mean']

# Create previous season mapping
cluster_season_prev = cluster_season_means.copy()
cluster_season_prev['id_season'] = cluster_season_prev['id_season'] + 1
cluster_season_prev.columns = ['emb_cluster', 'id_season', 'cluster_demand_last_season']

# Merge to get previous season demand
cluster_demand_last_season_df = df[['emb_cluster', 'id_season']].drop_duplicates().merge(
    cluster_season_prev[['emb_cluster', 'id_season', 'cluster_demand_last_season']],
    on=['emb_cluster', 'id_season'],
    how='left'
)

# Fill missing with cluster overall mean
cluster_demand_last_season_df = cluster_demand_last_season_df.merge(
    cluster_overall_means,
    on='emb_cluster',
    how='left'
)
cluster_demand_last_season_df['cluster_demand_last_season'] = cluster_demand_last_season_df['cluster_demand_last_season'].fillna(
    cluster_demand_last_season_df['cluster_overall_mean']
)
cluster_demand_last_season_df = cluster_demand_last_season_df[['emb_cluster', 'id_season', 'cluster_demand_last_season']]

if 'cluster_demand_last_season' in df.columns:
    df = df.drop(columns=['cluster_demand_last_season'])

df = df.merge(cluster_demand_last_season_df, on=['emb_cluster', 'id_season'], how='left')
if 'cluster_demand_last_season' in df.columns:
    cluster_overall_mean_dict = df.groupby('emb_cluster')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['cluster_demand_last_season'] = df['cluster_demand_last_season'].fillna(
        df['emb_cluster'].map(cluster_overall_mean_dict).fillna(overall_mean_value)
    )
else:
    cluster_overall_mean_dict = df.groupby('emb_cluster')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['cluster_demand_last_season'] = df['emb_cluster'].map(cluster_overall_mean_dict).fillna(overall_mean_value)

# Cluster demand slope across weeks (linear regression slope)
cluster_slopes = []
for cluster in df['emb_cluster'].unique():
    cluster_data = df[df['emb_cluster'] == cluster].copy()
    if len(cluster_data) > 1:
        weeks = cluster_data['weeks_since_launch'].values
        sales = cluster_data['weekly_sales'].values
        if len(weeks) > 1 and weeks.std() > 0:
            slope, _, _, _, _ = stats.linregress(weeks, sales)
        else:
            slope = 0
    else:
        slope = 0
    cluster_slopes.append({'emb_cluster': cluster, 'cluster_demand_slope': slope})

cluster_slopes_df = pd.DataFrame(cluster_slopes)
if 'cluster_demand_slope' in df.columns:
    df = df.drop(columns=['cluster_demand_slope'])

df = df.merge(cluster_slopes_df, on='emb_cluster', how='left')
df['cluster_demand_slope'] = df['cluster_demand_slope'].fillna(0)

# Cluster popularity (# unique products per cluster)
cluster_popularity = df.groupby('emb_cluster')['ID'].nunique().reset_index()
cluster_popularity.columns = ['emb_cluster', 'cluster_popularity']
if 'cluster_popularity' in df.columns:
    df = df.drop(columns=['cluster_popularity'])

df = df.merge(cluster_popularity, on='emb_cluster', how='left')

# Cluster season-on-season growth
print("Computing cluster season-on-season growth...")
cluster_season_means = df.groupby(['emb_cluster', 'id_season'])['weekly_sales'].mean().reset_index()
cluster_season_means.columns = ['emb_cluster', 'id_season', 'cluster_season_mean']
cluster_season_means = cluster_season_means.sort_values(['emb_cluster', 'id_season'])

cluster_season_means['prev_season_mean'] = cluster_season_means.groupby('emb_cluster')['cluster_season_mean'].shift(1)
cluster_season_means['cluster_season_growth'] = np.where(
    cluster_season_means['prev_season_mean'] > 0,
    (cluster_season_means['cluster_season_mean'] - cluster_season_means['prev_season_mean']) / 
    cluster_season_means['prev_season_mean'],
    np.where(cluster_season_means['cluster_season_mean'] > 0, 1.0, 0)
)
cluster_season_means['cluster_season_growth'] = cluster_season_means['cluster_season_growth'].fillna(0)

cluster_growth_df = cluster_season_means[['emb_cluster', 'id_season', 'cluster_season_growth']].copy()
if 'cluster_season_growth' in df.columns:
    df = df.drop(columns=['cluster_season_growth'])

df = df.merge(cluster_growth_df, on=['emb_cluster', 'id_season'], how='left')
df['cluster_season_growth'] = df['cluster_season_growth'].fillna(0)

# Cluster demand Y/Y change (year-over-year trend)
print("Computing cluster Y/Y change...")
df['season_type'] = (df['id_season'] % 2).astype(int)

cluster_yoy_means = df.groupby(['emb_cluster', 'year', 'season_type'])['weekly_sales'].mean().reset_index()
cluster_yoy_means.columns = ['emb_cluster', 'year', 'season_type', 'cluster_season_mean']

cluster_yoy_computed = df[['emb_cluster', 'ID', 'id_season', 'year', 'season_type']].copy()
cluster_yoy_computed = cluster_yoy_computed.merge(
    cluster_yoy_means, 
    on=['emb_cluster', 'year', 'season_type'], 
    how='left'
)

cluster_yoy_means_prev = cluster_yoy_means.copy()
cluster_yoy_means_prev['year'] = cluster_yoy_means_prev['year'] + 1
cluster_yoy_means_prev.columns = ['emb_cluster', 'year', 'season_type', 'cluster_season_mean_prev']
cluster_yoy_computed = cluster_yoy_computed.merge(
    cluster_yoy_means_prev[['emb_cluster', 'year', 'season_type', 'cluster_season_mean_prev']],
    on=['emb_cluster', 'year', 'season_type'],
    how='left'
)

cluster_yoy_computed['cluster_season_mean_prev'] = cluster_yoy_computed['cluster_season_mean_prev'].fillna(
    cluster_yoy_computed['cluster_season_mean']
)

cluster_yoy_computed['cluster_yoy_change'] = np.where(
    cluster_yoy_computed['cluster_season_mean_prev'] > 0,
    (cluster_yoy_computed['cluster_season_mean'] - cluster_yoy_computed['cluster_season_mean_prev']) / 
    cluster_yoy_computed['cluster_season_mean_prev'],
    0
)

cluster_yoy_df = cluster_yoy_computed[['emb_cluster', 'ID', 'id_season', 'cluster_yoy_change']].copy()

df = df.drop(columns=['season_type'])
if 'cluster_yoy_change' in df.columns:
    df = df.drop(columns=['cluster_yoy_change'])

df = df.merge(cluster_yoy_df, on=['emb_cluster', 'ID', 'id_season'], how='left')
df['cluster_yoy_change'] = df['cluster_yoy_change'].fillna(0)

# Cluster peak week
cluster_peak_week = df.groupby(['emb_cluster', 'weeks_since_launch'])['weekly_sales'].mean().reset_index()
cluster_peak_week = cluster_peak_week.loc[cluster_peak_week.groupby('emb_cluster')['weekly_sales'].idxmax()]
cluster_peak_week = cluster_peak_week[['emb_cluster', 'weeks_since_launch']].copy()
cluster_peak_week.columns = ['emb_cluster', 'cluster_peak_week']

if 'cluster_peak_week' in df.columns:
    df = df.drop(columns=['cluster_peak_week'])

df = df.merge(cluster_peak_week, on='emb_cluster', how='left')
df['cluster_peak_week'] = df['cluster_peak_week'].fillna(df['weeks_since_launch'].median())

# Merge velocity_1_6 to df
if 'cluster_velocity_1_6' in df.columns:
    df = df.drop(columns=['cluster_velocity_1_6'])

df = df.merge(cluster_velocity_1_6, on='emb_cluster', how='left')
overall_velocity_1_6 = train_first_6_weeks['weekly_sales'].mean()
df['cluster_velocity_1_6'] = df['cluster_velocity_1_6'].fillna(overall_velocity_1_6)

print(f"  ✓ Cluster velocity 1-6")
print(f"  ✓ Cluster demand last season")
print(f"  ✓ Cluster demand slope")
print(f"  ✓ Cluster popularity")
print(f"  ✓ Cluster season-on-season growth")
print(f"  ✓ Cluster Y/Y change")
print(f"  ✓ Cluster peak week")


Engineering Advanced Features for Train Data

1. Cluster-level features...
Computing cluster demand last season...
Computing cluster season-on-season growth...
Computing cluster Y/Y change...
  ✓ Cluster velocity 1-6
  ✓ Cluster demand last season
  ✓ Cluster demand slope
  ✓ Cluster popularity
  ✓ Cluster season-on-season growth
  ✓ Cluster Y/Y change
  ✓ Cluster peak week


In [9]:
# ========== SIMILARITY-TO-PREVIOUS-PRODUCTS FEATURES ==========
print("\n2. Similarity-to-previous-products features...")

# Get unique products with their embeddings (PCA features)
pca_cols = [f'emb_pca_{i+1}' for i in range(n_components)]
train_products_unique = df.groupby('ID').first().reset_index()

# Prepare embeddings for similarity computation
train_products_embeddings = train_products_unique[pca_cols].fillna(0).astype(np.float32).values

# Pre-compute product demand statistics
print("Pre-computing product statistics...")
product_stats = df.groupby('ID').agg({
    'weekly_sales': ['mean', 'median']
}).reset_index()
product_stats.columns = ['ID', 'product_demand_mean', 'product_demand_median']

product_velocity = df[df['weeks_since_launch'] < 3].groupby('ID')['weekly_sales'].mean().reset_index()
product_velocity.columns = ['ID', 'product_velocity_1_3']

overall_mean = df['weekly_sales'].mean()
overall_median = df['weekly_sales'].median()
overall_velocity = df[df['weeks_since_launch'] < 3]['weekly_sales'].mean()

# For each product, find similar products (using KNN on embeddings)
n_neighbors = min(10, len(train_products_unique) - 1)
batch_size = 100

if n_neighbors > 0 and len(train_products_unique) > 1:
    print(f"Computing similarities for {len(train_products_unique)} products in batches of {batch_size}...")
    knn = NearestNeighbors(n_neighbors=n_neighbors + 1, metric='cosine', algorithm='brute')
    knn.fit(train_products_embeddings)
    
    similar_product_features = []
    product_ids_array = train_products_unique['ID'].values
    
    for batch_start in range(0, len(train_products_unique), batch_size):
        batch_end = min(batch_start + batch_size, len(train_products_unique))
        batch_embeddings = train_products_embeddings[batch_start:batch_end]
        
        distances, indices = knn.kneighbors(batch_embeddings)
        
        for i, idx in enumerate(range(batch_start, batch_end)):
            product_id = product_ids_array[idx]
            similar_indices = indices[i][1:]
            similar_product_ids = product_ids_array[similar_indices]
            
            similar_stats = product_stats[product_stats['ID'].isin(similar_product_ids)]
            similar_velocities = product_velocity[product_velocity['ID'].isin(similar_product_ids)]
            
            if len(similar_stats) > 0:
                similar_demand_mean = similar_stats['product_demand_mean'].mean()
                similar_demand_median = similar_stats['product_demand_median'].median()
                similar_velocity_1_3 = similar_velocities['product_velocity_1_3'].mean() if len(similar_velocities) > 0 else similar_demand_mean
            else:
                similar_demand_mean = overall_mean
                similar_demand_median = overall_median
                similar_velocity_1_3 = overall_velocity
            
            similar_product_features.append({
                'ID': product_id,
                'similar_product_demand_mean': similar_demand_mean,
                'similar_product_demand_median': similar_demand_median,
                'similar_product_velocity_1_3': similar_velocity_1_3
            })
        
        if (batch_start // batch_size + 1) % 10 == 0:
            print(f"  Processed {batch_end}/{len(train_products_unique)} products...")
    
    similar_features_df = pd.DataFrame(similar_product_features)
    
    for col in ['similar_product_demand_mean', 'similar_product_demand_median', 'similar_product_velocity_1_3']:
        if col in df.columns:
            df = df.drop(columns=[col])
    
    df = df.merge(similar_features_df, on='ID', how='left')
    df['similar_product_demand_mean'] = df['similar_product_demand_mean'].fillna(df['weekly_sales'].mean())
    df['similar_product_demand_median'] = df['similar_product_demand_median'].fillna(df['weekly_sales'].median())
    df['similar_product_velocity_1_3'] = df['similar_product_velocity_1_3'].fillna(
        df[df['weeks_since_launch'] < 3]['weekly_sales'].mean()
    )
else:
    df['similar_product_demand_mean'] = overall_mean
    df['similar_product_demand_median'] = overall_median
    df['similar_product_velocity_1_3'] = overall_velocity

print(f"  ✓ Similar product demand mean")
print(f"  ✓ Similar product demand median")
print(f"  ✓ Similar product velocity 1-3")


2. Similarity-to-previous-products features...
Pre-computing product statistics...
Computing similarities for 9843 products in batches of 100...
  Processed 1000/9843 products...
  Processed 2000/9843 products...
  Processed 3000/9843 products...
  Processed 4000/9843 products...
  Processed 5000/9843 products...
  Processed 6000/9843 products...
  Processed 7000/9843 products...
  Processed 8000/9843 products...
  Processed 9000/9843 products...
  ✓ Similar product demand mean
  ✓ Similar product demand median
  ✓ Similar product velocity 1-3


In [10]:
# ========== FAMILY-LEVEL TREND FEATURES ==========
print("\n3. Family-level trend features...")

# Family velocity 1-3 last season
print("Computing family velocity 1-3 last season...")
family_first_3 = df[df['weeks_since_launch'] < 3]
family_season_velocity = family_first_3.groupby(['family', 'id_season'])['weekly_sales'].mean().reset_index()
family_season_velocity.columns = ['family', 'id_season', 'family_velocity_1_3']

family_overall_velocity_stats = family_first_3.groupby('family')['weekly_sales'].mean().reset_index()
family_overall_velocity_stats.columns = ['family', 'family_overall_velocity']

family_velocity_prev = family_season_velocity.copy()
family_velocity_prev['id_season'] = family_velocity_prev['id_season'] + 1
family_velocity_prev.columns = ['family', 'id_season', 'family_velocity_1_3_last_season']

family_velocity_last_season_df = df[['family', 'id_season']].drop_duplicates().merge(
    family_velocity_prev[['family', 'id_season', 'family_velocity_1_3_last_season']],
    on=['family', 'id_season'],
    how='left'
)

family_velocity_last_season_df = family_velocity_last_season_df.merge(
    family_overall_velocity_stats,
    on='family',
    how='left'
)
family_velocity_last_season_df['family_velocity_1_3_last_season'] = family_velocity_last_season_df['family_velocity_1_3_last_season'].fillna(
    family_velocity_last_season_df['family_overall_velocity']
)
family_velocity_last_season_df = family_velocity_last_season_df[['family', 'id_season', 'family_velocity_1_3_last_season']]

if 'family_velocity_1_3_last_season' in df.columns:
    df = df.drop(columns=['family_velocity_1_3_last_season'])

df = df.merge(family_velocity_last_season_df, on=['family', 'id_season'], how='left')
if 'family_velocity_1_3_last_season' in df.columns:
    family_overall_velocity = df[df['weeks_since_launch'] < 3].groupby('family')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['family_velocity_1_3_last_season'] = df['family_velocity_1_3_last_season'].fillna(
        df['family'].map(family_overall_velocity).fillna(overall_mean_value)
    )
else:
    family_overall_velocity = df[df['weeks_since_launch'] < 3].groupby('family')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['family_velocity_1_3_last_season'] = df['family'].map(family_overall_velocity).fillna(overall_mean_value)

# Family demand mean last season
print("Computing family demand mean last season...")
family_season_means = df.groupby(['family', 'id_season'])['weekly_sales'].mean().reset_index()
family_season_means.columns = ['family', 'id_season', 'family_season_mean']

family_overall_means = df.groupby('family')['weekly_sales'].mean().reset_index()
family_overall_means.columns = ['family', 'family_overall_mean']

family_season_prev = family_season_means.copy()
family_season_prev['id_season'] = family_season_prev['id_season'] + 1
family_season_prev.columns = ['family', 'id_season', 'family_demand_mean_last_season']

family_demand_last_season_df = df[['family', 'id_season']].drop_duplicates().merge(
    family_season_prev[['family', 'id_season', 'family_demand_mean_last_season']],
    on=['family', 'id_season'],
    how='left'
)

family_demand_last_season_df = family_demand_last_season_df.merge(
    family_overall_means,
    on='family',
    how='left'
)
family_demand_last_season_df['family_demand_mean_last_season'] = family_demand_last_season_df['family_demand_mean_last_season'].fillna(
    family_demand_last_season_df['family_overall_mean']
)
family_demand_last_season_df = family_demand_last_season_df[['family', 'id_season', 'family_demand_mean_last_season']]

if 'family_demand_mean_last_season' in df.columns:
    df = df.drop(columns=['family_demand_mean_last_season'])

df = df.merge(family_demand_last_season_df, on=['family', 'id_season'], how='left')
if 'family_demand_mean_last_season' in df.columns:
    family_overall_mean = df.groupby('family')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['family_demand_mean_last_season'] = df['family_demand_mean_last_season'].fillna(
        df['family'].map(family_overall_mean).fillna(overall_mean_value)
    )
else:
    family_overall_mean = df.groupby('family')['weekly_sales'].mean().to_dict()
    overall_mean_value = df['weekly_sales'].mean()
    df['family_demand_mean_last_season'] = df['family'].map(family_overall_mean).fillna(overall_mean_value)

# Family demand trend (slope over last 4 seasons)
print("Computing family demand trend...")
family_season_means = df.groupby(['family', 'id_season'])['weekly_sales'].mean().reset_index()
family_season_means.columns = ['family', 'id_season', 'family_season_mean']
family_season_means = family_season_means.sort_values(['family', 'id_season'])

family_trends = []
for family in family_season_means['family'].unique():
    family_seasons = family_season_means[family_season_means['family'] == family].sort_values('id_season')
    recent_seasons = family_seasons.tail(4)
    
    if len(recent_seasons) > 1:
        x = np.arange(len(recent_seasons))
        y = recent_seasons['family_season_mean'].values
        slope, _, _, _, _ = stats.linregress(x, y)
    else:
        slope = 0
    
    family_trends.append({'family': family, 'family_demand_trend': slope})

family_trend_map = pd.DataFrame(family_trends)

family_trend_df = df[['family', 'id_season']].drop_duplicates().merge(
    family_trend_map,
    on='family',
    how='left'
)
family_trend_df['family_demand_trend'] = family_trend_df['family_demand_trend'].fillna(0)

if 'family_demand_trend' in df.columns:
    df = df.drop(columns=['family_demand_trend'])

df = df.merge(family_trend_df, on=['family', 'id_season'], how='left')
df['family_demand_trend'] = df['family_demand_trend'].fillna(0)

print(f"  ✓ Family velocity 1-3 last season")
print(f"  ✓ Family demand mean last season")
print(f"  ✓ Family demand trend")

print("\n" + "=" * 60)
print("Advanced feature engineering for train data complete!")
print("=" * 60)


3. Family-level trend features...
Computing family velocity 1-3 last season...
Computing family demand mean last season...
Computing family demand trend...
  ✓ Family velocity 1-3 last season
  ✓ Family demand mean last season
  ✓ Family demand trend

Advanced feature engineering for train data complete!


### 6.4 Apply Advanced Features to Test Data

Test data doesn't have sales, so we use train-based statistics computed in section 6.3.

In [11]:
# Apply cluster features to test data
print("=" * 60)
print("Applying Advanced Features to Test Data")
print("=" * 60)

print("\n1. Adding cluster features to test data...")

# Cluster velocity 1-6
df_test = df_test.merge(cluster_velocity_1_6, on='emb_cluster', how='left')
overall_velocity_1_6 = train_first_6_weeks['weekly_sales'].mean()
df_test['cluster_velocity_1_6'] = df_test['cluster_velocity_1_6'].fillna(overall_velocity_1_6)

# Cluster demand last season - use last season from train
latest_season = df['id_season'].max()
cluster_latest_season_demand = df[df['id_season'] == latest_season].groupby('emb_cluster')['weekly_sales'].mean().reset_index()
cluster_latest_season_demand.columns = ['emb_cluster', 'cluster_demand_last_season']
df_test = df_test.merge(cluster_latest_season_demand, on='emb_cluster', how='left')
cluster_overall_mean = df.groupby('emb_cluster')['weekly_sales'].mean().to_dict()
df_test['cluster_demand_last_season'] = df_test.apply(
    lambda row: cluster_overall_mean.get(row['emb_cluster'], df['weekly_sales'].mean()) 
    if pd.isna(row['cluster_demand_last_season']) 
    else row['cluster_demand_last_season'], axis=1
)

# Cluster demand slope, popularity, growth, Y/Y change, peak week
df_test = df_test.merge(cluster_slopes_df, on='emb_cluster', how='left')
df_test['cluster_demand_slope'] = df_test['cluster_demand_slope'].fillna(0)

df_test = df_test.merge(cluster_popularity, on='emb_cluster', how='left')
df_test['cluster_popularity'] = df_test['cluster_popularity'].fillna(0)

latest_growth = cluster_growth_df[cluster_growth_df['id_season'] == latest_season].copy()
if len(latest_growth) > 0:
    latest_growth = latest_growth[['emb_cluster', 'cluster_season_growth']]
    df_test = df_test.merge(latest_growth, on='emb_cluster', how='left')
else:
    df_test['cluster_season_growth'] = 0
df_test['cluster_season_growth'] = df_test['cluster_season_growth'].fillna(0)

latest_yoy = cluster_yoy_df[cluster_yoy_df['id_season'] == latest_season].groupby('emb_cluster')['cluster_yoy_change'].mean().reset_index()
df_test = df_test.merge(latest_yoy, on='emb_cluster', how='left')
df_test['cluster_yoy_change'] = df_test['cluster_yoy_change'].fillna(0)

df_test = df_test.merge(cluster_peak_week, on='emb_cluster', how='left')
df_test['cluster_peak_week'] = df_test['cluster_peak_week'].fillna(df['weeks_since_launch'].median() if 'weeks_since_launch' in df_test.columns else 10)

print(f"  ✓ All cluster features added")

Applying Advanced Features to Test Data

1. Adding cluster features to test data...
  ✓ All cluster features added


In [12]:
# Apply similarity features to test data
print("\n2. Computing similarity features for test products...")

test_products_unique = df_test.groupby('ID').first().reset_index()
test_products_embeddings = test_products_unique[pca_cols].fillna(0).values

if len(test_products_unique) > 0 and len(train_products_unique) > 0:
    knn_test = NearestNeighbors(n_neighbors=min(10, len(train_products_unique)), metric='cosine')
    knn_test.fit(train_products_embeddings)
    
    test_similar_features = []
    
    for idx, test_product_row in test_products_unique.iterrows():
        test_product_id = test_product_row['ID']
        test_product_embedding = test_products_embeddings[idx:idx+1]
        distances, indices = knn_test.kneighbors(test_product_embedding)
        
        similar_indices = indices[0]
        similar_product_ids = train_products_unique.iloc[similar_indices]['ID'].values
        similar_products_data = df[df['ID'].isin(similar_product_ids)]
        
        if len(similar_products_data) > 0:
            similar_demand_mean = similar_products_data['weekly_sales'].mean()
            similar_demand_median = similar_products_data['weekly_sales'].median()
            similar_first_3_weeks = similar_products_data[similar_products_data['weeks_since_launch'] < 3]
            similar_velocity_1_3 = similar_first_3_weeks['weekly_sales'].mean() if len(similar_first_3_weeks) > 0 else similar_demand_mean
        else:
            similar_demand_mean = df['weekly_sales'].mean()
            similar_demand_median = df['weekly_sales'].median()
            similar_velocity_1_3 = df[df['weeks_since_launch'] < 3]['weekly_sales'].mean()
        
        test_similar_features.append({
            'ID': test_product_id,
            'similar_product_demand_mean': similar_demand_mean,
            'similar_product_demand_median': similar_demand_median,
            'similar_product_velocity_1_3': similar_velocity_1_3
        })
    
    test_similar_features_df = pd.DataFrame(test_similar_features)
    df_test = df_test.merge(test_similar_features_df, on='ID', how='left')
    df_test['similar_product_demand_mean'] = df_test['similar_product_demand_mean'].fillna(df['weekly_sales'].mean())
    df_test['similar_product_demand_median'] = df_test['similar_product_demand_median'].fillna(df['weekly_sales'].median())
    df_test['similar_product_velocity_1_3'] = df_test['similar_product_velocity_1_3'].fillna(
        df[df['weeks_since_launch'] < 3]['weekly_sales'].mean()
    )
else:
    overall_mean = df['weekly_sales'].mean()
    overall_median = df['weekly_sales'].median()
    overall_velocity = df[df['weeks_since_launch'] < 3]['weekly_sales'].mean()
    df_test['similar_product_demand_mean'] = overall_mean
    df_test['similar_product_demand_median'] = overall_median
    df_test['similar_product_velocity_1_3'] = overall_velocity

print(f"  ✓ Similarity features computed using train data")



2. Computing similarity features for test products...
  ✓ Similarity features computed using train data


In [13]:
# Apply family features to test data
print("\n3. Adding family features to test data...")

family_latest_velocity = family_velocity_last_season_df[family_velocity_last_season_df['id_season'] == latest_season].copy()
if len(family_latest_velocity) > 0:
    family_latest_velocity = family_latest_velocity[['family', 'family_velocity_1_3_last_season']]
    df_test = df_test.merge(family_latest_velocity, on='family', how='left')
else:
    df_test['family_velocity_1_3_last_season'] = df_test['family'].map(family_overall_velocity).fillna(df['weekly_sales'].mean())
df_test['family_velocity_1_3_last_season'] = df_test['family_velocity_1_3_last_season'].fillna(
    df_test['family'].map(family_overall_velocity).fillna(df['weekly_sales'].mean())
)

family_latest_demand = family_demand_last_season_df[family_demand_last_season_df['id_season'] == latest_season].copy()
if len(family_latest_demand) > 0:
    family_latest_demand = family_latest_demand[['family', 'family_demand_mean_last_season']]
    df_test = df_test.merge(family_latest_demand, on='family', how='left')
else:
    df_test['family_demand_mean_last_season'] = df_test['family'].map(family_overall_mean).fillna(df['weekly_sales'].mean())
df_test['family_demand_mean_last_season'] = df_test['family_demand_mean_last_season'].fillna(
    df_test['family'].map(family_overall_mean).fillna(df['weekly_sales'].mean())
)

family_latest_trend = family_trend_df[family_trend_df['id_season'] == latest_season].copy()
if len(family_latest_trend) > 0:
    family_latest_trend = family_latest_trend[['family', 'family_demand_trend']]
    df_test = df_test.merge(family_latest_trend, on='family', how='left')
else:
    df_test['family_demand_trend'] = 0
df_test['family_demand_trend'] = df_test['family_demand_trend'].fillna(0)

print(f"  ✓ All family features added")


3. Adding family features to test data...
  ✓ All family features added


## 7. Final Processing and Save

### 7.1 Remove Low Importance Features

In [14]:
# Remove low importance features (based on feature importance analysis)
low_importance_features = [
    "cluster_velocity_1_6",
    "cluster_peak_week",
    "family_demand_trend",
    "cluster_popularity",
    "cluster_demand_slope",
    "cluster_season_growth",
    "aggregated_family",  # if exists
    "cluster_yoy_change",
]

features_to_remove = [f for f in low_importance_features if f in df.columns]
if features_to_remove:
    df.drop(columns=features_to_remove, inplace=True)
    print(f"Removed {len(features_to_remove)} low importance features from train: {features_to_remove}")

features_to_remove_test = [f for f in low_importance_features if f in df_test.columns]
if features_to_remove_test:
    df_test.drop(columns=features_to_remove_test, inplace=True)
    print(f"Removed {len(features_to_remove_test)} low importance features from test: {features_to_remove_test}")


Removed 8 low importance features from train: ['cluster_velocity_1_6', 'cluster_peak_week', 'family_demand_trend', 'cluster_popularity', 'cluster_demand_slope', 'cluster_season_growth', 'aggregated_family', 'cluster_yoy_change']
Removed 8 low importance features from test: ['cluster_velocity_1_6', 'cluster_peak_week', 'family_demand_trend', 'cluster_popularity', 'cluster_demand_slope', 'cluster_season_growth', 'aggregated_family', 'cluster_yoy_change']


### 7.2 Save Processed Data

In [15]:
# Create data/processed directory if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

# Exclude embedding columns (for faster save/load)
embedding_cols_to_exclude = ['image_embedding', 'embedding_array'] + [f'emb_pca_{i+1}' for i in range(n_components)]
train_cols_to_save = [col for col in df.columns if col not in embedding_cols_to_exclude]
test_cols_to_save = [col for col in df_test.columns if col not in embedding_cols_to_exclude]

# Save processed dataframes
df[train_cols_to_save].to_csv('data/processed/train_processed.csv', index=False)
df_test[test_cols_to_save].to_csv('data/processed/test_processed.csv', index=False)

print("=" * 60)
print("✓ data/processed/train_processed.csv saved successfully!")
print("✓ data/processed/test_processed.csv saved successfully!")
print(f"  Excluded {len(embedding_cols_to_exclude)} embedding columns for faster processing")
print("=" * 60)
print("\nAll feature engineering complete!")

✓ data/processed/train_processed.csv saved successfully!
✓ data/processed/test_processed.csv saved successfully!
  Excluded 85 embedding columns for faster processing

All feature engineering complete!
